In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kstest, uniform

In [ ]:
DATASETS = ['kin8nm', 'house_8L', 'puma8NH', 'BNG_stock', 'cmc']
METHODS = ['TabPFN', 'GP']
N_REP = 20

RAW_DIR = 'output_realdata/raw'

# Load all results
results = {}  # results[dataset][seed] = output dict
for ds in DATASETS:
    results[ds] = {}
    for seed in range(1, N_REP + 1):
        path = os.path.join(RAW_DIR, f'{ds}_seed{seed}.pickle')
        if os.path.exists(path):
            with open(path, 'rb') as f:
                results[ds][seed] = pickle.load(f)

print('Loaded reps per dataset:')
for ds in DATASETS:
    print(f'  {ds}: {len(results[ds])} / {N_REP}')

## 1. Calibration Plot (Reliability Diagram)
Plot empirical coverage vs. nominal coverage for each method, averaged across datasets and repetitions.

In [ ]:
# Collect coverage arrays across all datasets and reps
# Shape: [method -> list of (n_pi_levels,) arrays]
all_coverage = {m: [] for m in METHODS}

for ds in DATASETS:
    for seed, res in results[ds].items():
        pi_levels = res['pi_levels']
        for m in METHODS:
            all_coverage[m].append(res[f'{m}_coverage'])

pi_levels = results[DATASETS[0]][1]['pi_levels']

fig, ax = plt.subplots(figsize=(5, 5))
colors = {'TabPFN': '#1f77b4', 'GP': '#ff7f0e'}
markers = {'TabPFN': 'o', 'GP': 's'}

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect calibration')

for m in METHODS:
    cov_arr = np.array(all_coverage[m])  # (n_reps * n_datasets, n_pi_levels)
    mean_cov = cov_arr.mean(axis=0)
    sd_cov = cov_arr.std(axis=0)
    ax.errorbar(pi_levels, mean_cov, yerr=sd_cov,
                label=m, color=colors[m], marker=markers[m],
                capsize=3, lw=1.5)

ax.set_xlabel('Nominal coverage level', fontsize=12)
ax.set_ylabel('Empirical coverage', fontsize=12)
ax.set_title('Calibration plot (real data)', fontsize=13)
ax.legend()
ax.set_xlim(0.45, 1.0)
ax.set_ylim(0.45, 1.0)
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig('output_realdata/calibration_plot.pdf', bbox_inches='tight')
plt.show()

## 2. PIT Histogram
Pool all PIT values across datasets and repetitions. A flat histogram indicates calibration.

In [ ]:
all_pit = {m: [] for m in METHODS}

for ds in DATASETS:
    for seed, res in results[ds].items():
        for m in METHODS:
            all_pit[m].extend(res[f'{m}_pit'].tolist())

fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharey=True)

for ax, m in zip(axes, METHODS):
    pit_vals = np.array(all_pit[m])
    ax.hist(pit_vals, bins=20, density=True, color=colors[m], alpha=0.7,
            edgecolor='white')
    ax.axhline(1.0, color='k', lw=1, ls='--', label='Uniform')
    ax.set_xlabel('PIT value', fontsize=11)
    ax.set_title(m, fontsize=12)

axes[0].set_ylabel('Density', fontsize=11)
axes[0].legend()
fig.suptitle('PIT histograms (pooled across datasets and repetitions)', fontsize=12)
plt.tight_layout()
plt.savefig('output_realdata/pit_histogram.pdf', bbox_inches='tight')
plt.show()

## 3. Summary Table
Per-dataset metrics: Coverage@95%, mean interval width at 95%, CRPS, KS-PIT statistic.

In [ ]:
rows = []
pi_idx_95 = pi_levels.index(0.95)

for ds in DATASETS:
    reps = results[ds]
    if not reps:
        continue
    for m in METHODS:
        cov95 = np.mean([r[f'{m}_coverage'][pi_idx_95] for r in reps.values()])
        width = np.mean([r[f'{m}_width_95'] for r in reps.values()])
        crps = np.mean([r[f'{m}_crps'] for r in reps.values()])
        pit_vals = np.concatenate([r[f'{m}_pit'] for r in reps.values()])
        ks_stat, _ = kstest(pit_vals, uniform(0, 1).cdf)
        rows.append({
            'Dataset': ds,
            'Method': m,
            'Coverage@95%': round(cov95, 3),
            'Width@95%': round(width, 3),
            'CRPS': round(crps, 4),
            'KS-PIT': round(ks_stat, 4),
        })

df_summary = pd.DataFrame(rows)
display(df_summary.pivot_table(
    index='Dataset', columns='Method',
    values=['Coverage@95%', 'Width@95%', 'CRPS', 'KS-PIT']
).round(3))

# Also print averaged across datasets
print('\nAveraged across all datasets:')
display(df_summary.groupby('Method')[['Coverage@95%', 'Width@95%', 'CRPS', 'KS-PIT']].mean().round(3))

In [ ]:
# LaTeX table for the paper
print(df_summary.to_latex(index=False, float_format='%.3f'))